In [ ]:
# !pip install pandas pyarrow anthropic

In [1]:
import anthropic

anthropic.__version__

'0.49.0'

In [2]:
import pandas as pd

DATA_FILE = "data/ml_interview_dataset.parquet"
df = pd.read_parquet(DATA_FILE)
df["delivery_date"] = pd.to_datetime(df["delivery_date"])
df["ordered_date"] = pd.to_datetime(df["ordered_date"])

print(df.head())

                               order_id department_name         vendor_name  \
0  e811cda0-e90c-4553-890f-20762f7f35c2     Electronics  TechSuppliers Inc.   
1  ec44a8f4-c814-4ec6-958d-e17508b05302     Maintenance      OfficeMax Corp   
2  e0ccaef8-de5b-4406-8ad4-0d11af76be46     Maintenance       GlobalProcure   
3  3f8b5e2b-436a-402e-8e65-bdbb58bfb9a3         Kitchen       GlobalProcure   
4  e8387886-3155-4e13-a045-aeb3de65c1d1       Furniture     SupplyChain Co.   

    item_name  item_quantity  unit_cost  total_cost        ordered_by  \
0    Keyboard              2     341.58      683.16  Jeffrey Lawrence   
1       Tools              2     117.13      234.26  Maria Montgomery   
2  Repair Kit              9     215.56     1940.04  Danielle Johnson   
3    Utensils              3     352.09     1056.27  Jeffrey Lawrence   
4        Sofa              2      55.45      110.90    Robert Johnson   

         ordered_date        approved_by       delivery_date  
0 2024-08-17 20:10:09  

In [ ]:
import pandas as pd
import numpy as np
from datetime import date
import anthropic


client = anthropic.Anthropic()

class SpendAssistant:
    def __init__(self, model="claude-3-7-sonnet-20250219", debug=False):
        self.debug = debug
        self.model = model

    def _log_debug(self, label, content):
        if self.debug:
            print(f"\n[DEBUG] {label}:\n{content}\n")

    def _chat(self, system_prompt, user_prompt):
        message = client.messages.create(
            model=self.model,
            system=system_prompt,
            messages=[
                {"role": "user", "content": [{"type": "text", "text": user_prompt}]}
            ],
            max_tokens=1000,
            temperature=0,
        )
        answer = message.content[0].text
        return answer

    def get_pandas_expression(self, user_query, df_head):
        today = date.today().strftime("%B %d, %Y")
        system_prompt = (
            "You are working with a pandas dataframe in Python. "
            "The name of the dataframe is `df`.\n"
            f"This is the result of `print(df.head())`:\n{df_head}\n\n"
            "Follow these instructions:\n"
            "Convert the query to executable Python code using Pandas and Numpy. "
            "The code should represent a solution to the query."
            "The final line of code should be a Python expression that can be called with the `eval()` function.\n"
            f"If no day or month or year are mentioned, assume the current day, month and year respectively. Current date is {today}.\n"
            "If the query asks for a report based on time periods, such as daily, monthly, quarterly, annual, "
            "   generate a complete breakdown for each requested period.\n"
            "If the query involves identifying a maximum/minimum or calculating a value (e.g. total, sum, average), "
            "   return both the label and its numeric value.\n"
            "Generate a code that avoids execution errors or runtime errors such as division by zero\n"
            "Separate the statements with only semicolons.\n"
        )
        user_prompt = (
            "PRINT ONLY THE EXPRESSION.\n"
            "Do not quote the expression.\n\n"
            f"Query: {user_query}\n\n"
            "Expression:"
        )
        return self._chat(system_prompt, user_prompt)

    def generate_final_response(self, user_query, pandas_expression, query_results):
        system_prompt = (
            "You are a Spend Assistant providing meaningful, insightful, and concise responses based on spending data."
        )
        user_prompt = (
            f"Given a user question, synthesize a response from the query results.\n"
            "If results are 'Empty', acknowledge there are no results.\n"
            "If results are 'Error', respond with: 'I cannot answer this question.'\n"
            f"User question: {user_query}\n"
            f"Pandas expression: ```{pandas_expression}```\n"
            f"Question results: {query_results}\n\n"
            "Response:"
        )
        return self._chat(system_prompt, user_prompt)

    def execute_expression(self, expression, df):
        local_vars = {'pd': pd, 'np': np, 'df': df}
        try:
            statements = [s.strip() for s in expression.replace('\n', ';').split(';') if s.strip()]
            for stmt in statements[:-1]:
                exec(stmt, {}, local_vars)
            result = eval(statements[-1], {}, local_vars)

            # Empty results
            if result is None or (not isinstance(result, (pd.DataFrame, pd.Series, np.ndarray, list)) and pd.isna(result)):
                return "Empty"
            
            return result
        
        except ValueError as ve:
            self._log_debug("Execution ValueError", str(ve))
            return "Empty"
        
        except Exception as e:
            self._log_debug("Execution Error", str(e))
            return "Error"

    def search(self, user_query, df):
        self._log_debug("Question", user_query)

        expr = self.get_pandas_expression(user_query, df.head())

        self._log_debug("Execution Expression", expr)
        
        results = self.execute_expression(expr, df)
        
        self._log_debug("Search Results", results)
        
        return expr, results

    def answer(self, user_query, df):
        pandas_expr, results = self.search(user_query, df)
        final_response = self.generate_final_response(user_query, pandas_expr, results)

        self._log_debug("Final Response", final_response)
        
        return final_response

In [5]:
spend_assistant = SpendAssistant(debug=True)

In [6]:
user_query = "Which department spent the most in 2023?"
response = spend_assistant.answer(user_query, df)


[DEBUG] Question:
Which department spent the most in 2023?


[DEBUG] Execution Expression:
df[df['ordered_date'].dt.year == 2023].groupby('department_name')['total_cost'].sum().sort_values(ascending=False).head(1)


[DEBUG] Search Results:
department_name
Maintenance    123507.78
Name: total_cost, dtype: float64


[DEBUG] Final Response:
The Maintenance department spent the most in 2023, with a total expenditure of $123,507.78.



In [7]:
user_query = "Who were the top 3 vendors for the department that spend the most?"
response = spend_assistant.answer(user_query, df)


[DEBUG] Question:
Who were the top 3 vendors for the department that spend the most?


[DEBUG] Execution Expression:
df.groupby('department_name')['total_cost'].sum().reset_index().sort_values('total_cost', ascending=False).iloc[0]['department_name']; top_dept = df.groupby('department_name')['total_cost'].sum().reset_index().sort_values('total_cost', ascending=False).iloc[0]['department_name']; df[df['department_name'] == top_dept].groupby('vendor_name')['total_cost'].sum().reset_index().sort_values('total_cost', ascending=False).head(3)


[DEBUG] Search Results:
      vendor_name  total_cost
4  OfficeMax Corp    43003.27
3      MegaVendor    36101.27
0     ElectroTech    35230.59


[DEBUG] Final Response:
Based on the spending data, the top 3 vendors for the department with the highest total spend are:

1. OfficeMax Corp ($43,003.27)
2. MegaVendor ($36,101.27)
3. ElectroTech ($35,230.59)

These vendors represent the largest spending relationships for the department with the highest o

In [8]:
user_query = "What is my total monthly spend?"
response = spend_assistant.answer(user_query, df)


[DEBUG] Question:
What is my total monthly spend?


[DEBUG] Execution Expression:
df['ordered_date'] = pd.to_datetime(df['ordered_date']); df['month_year'] = df['ordered_date'].dt.to_period('M'); monthly_spend = df.groupby('month_year')['total_cost'].sum().reset_index(); monthly_spend.rename(columns={'month_year': 'Month-Year', 'total_cost': 'Total Spend'}, inplace=True); monthly_spend


[DEBUG] Search Results:
   Month-Year  Total Spend
0     2023-03     22726.32
1     2023-04     48214.83
2     2023-05     56042.78
3     2023-06     58842.88
4     2023-07     54117.38
5     2023-08     62244.16
6     2023-09     67863.41
7     2023-10     77359.80
8     2023-11     51144.98
9     2023-12     48259.30
10    2024-01     48231.75
11    2024-02     60406.52
12    2024-03     45859.41
13    2024-04     66457.39
14    2024-05     54869.04
15    2024-06     58272.80
16    2024-07     62506.55
17    2024-08     73798.95
18    2024-09     55271.95
19    2024-10     42967.47
20    2024-11    

In [9]:
user_query = "Who were the top 3 approvers?"
response = spend_assistant.answer(user_query, df)


[DEBUG] Question:
Who were the top 3 approvers?


[DEBUG] Execution Expression:
df['approved_by'].value_counts().head(3)


[DEBUG] Search Results:
approved_by
Lindsay Blair    68
Lisa Smith       57
Jill Rhodes      55
Name: count, dtype: int64


[DEBUG] Final Response:
The top 3 approvers were:

1. Lindsay Blair with 68 approvals
2. Lisa Smith with 57 approvals 
3. Jill Rhodes with 55 approvals



In [10]:
user_query = "What are the top 3 departments by total spend?"
response = spend_assistant.answer(user_query, df)


[DEBUG] Question:
What are the top 3 departments by total spend?


[DEBUG] Execution Expression:
df.groupby('department_name')['total_cost'].sum().sort_values(descending=True).head(3)


[DEBUG] Execution Error:
Series.sort_values() got an unexpected keyword argument 'descending'


[DEBUG] Search Results:
Error


[DEBUG] Final Response:
I cannot answer this question.

